# Ingest — organisation attributes

Reads the Entra user export from `Files/landing/org/` into `org_attributes`.

This is what turns per-person consumption into a department, cost-centre or business-unit view —
which is most of the chargeback story. Without it the report still works; you just cannot group by
anything.

## Two things make this harder than it looks

**Every tenant spells these differently.** One exports `department`, another `Department`, another
`Organization`. So columns are matched against a list of known aliases rather than by exact name,
and anything expected but absent is created empty. One missing attribute should cost you that one
breakdown, not the whole table.

**Half of them are not standard Entra properties.** `department`, `jobTitle`, `city` and `country`
come out of the bulk download. `manager` needs Graph or the preview download. `costCenter`,
`jobFamily` and `businessUnit` are extension attributes whose names differ per tenant — usually
populated by an HR sync. If you do not have them, leave them out; nothing breaks.

**Full replace, not a merge.** This is current state, not history: someone who changed department
last month should appear once, in their new one. A merge would leave the old row behind.

In [ ]:
LANDING = "Files/landing/org"
TBL = "org_attributes"

# Canonical name -> the spellings seen in the wild. Add yours if it is missing;
# comparison ignores case, spaces, dashes and underscores.
ALIASES = {
    "user_principal_name": ["userprincipalname", "upn", "userprincipal",
                            "email", "mail", "emailaddress"],
    "display_name":        ["displayname", "name", "fullname", "preferredname"],
    "department":          ["department", "dept", "organization", "organisation"],
    "job_title":           ["jobtitle", "title", "role"],
    "job_family":          ["jobfamily", "function", "functiontype", "jobfunction"],
    "city":                ["city", "officelocation", "location"],
    "country":             ["country", "countryorregion", "region"],
    "cost_center":         ["costcenter", "costcentre"],
    "manager":             ["manager", "managername", "supervisor", "managerid",
                            "managerupn"],
    "business_unit":       ["businessunit", "division", "segment"],
}

In [ ]:
from pyspark.sql import functions as F
import re


def norm(name):
    return re.sub(r"[ _\-]", "", name).lower()


raw = (spark.read.option("header", True).option("inferSchema", False)
       .csv(f"{LANDING}/*.csv"))

print(f"read {raw.count():,} rows")
print("columns found:", raw.columns)

lookup = {norm(c): c for c in raw.columns}
resolved, missing = {}, []
for canon, alts in ALIASES.items():
    hit = next((lookup[norm(a)] for a in alts if norm(a) in lookup), None)
    if hit:
        resolved[canon] = hit
    else:
        missing.append(canon)

print()
for c, src in resolved.items():
    print(f"  {c:22s} <- {src}")
if missing:
    print()
    print(f"  not present, will be empty: {', '.join(missing)}")
    print("  (each one costs you that breakdown and nothing else)")

if "user_principal_name" not in resolved:
    raise ValueError(
        "No UPN column found. This is the join key to every consumption export - "
        f"without it the table cannot be used. Columns present: {raw.columns}")

In [ ]:
cols = []
for canon in ALIASES:
    if canon in resolved:
        # UPN is the join key, so it is lowercased and trimmed. A trailing space
        # or a capitalised domain is enough to make every join silently miss.
        e = F.trim(F.col(resolved[canon]))
        if canon == "user_principal_name":
            e = F.lower(e)
        cols.append(e.cast("string").alias(canon))
    else:
        cols.append(F.lit(None).cast("string").alias(canon))

org = (raw.select(*cols)
       .filter(F.col("user_principal_name").isNotNull()
               & (F.col("user_principal_name") != ""))
       .dropDuplicates(["user_principal_name"])
       .withColumn("_loaded_at", F.current_timestamp()))

print(f"{org.count():,} people after de-duplicating on UPN")

org.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(TBL)
print(f"{TBL}: written")

## Check the join will actually work

The most common disappointment with this template is department breakdowns coming back empty, and
the cause is almost always that the org file's UPNs do not match the consumption export's. Better
to find that here than to stare at a blank chart.

In [ ]:
spark.sql(f"""
    SELECT  COUNT(*)                                        AS people,
            COUNT(department)                               AS with_department,
            COUNT(cost_center)                              AS with_cost_centre,
            COUNT(manager)                                  AS with_manager,
            COUNT(DISTINCT department)                      AS departments
    FROM    {TBL}
""").show(truncate=False)

if spark.catalog.tableExists("viva_credits_weekly"):
    spark.sql(f"""
        WITH consumers AS (
            SELECT DISTINCT LOWER(TRIM(COALESCE(user_principal_name, person_id))) AS upn
            FROM   viva_credits_weekly
        )
        SELECT  COUNT(*)                                   AS consumers,
                COUNT(o.user_principal_name)               AS matched_to_org,
                ROUND(100.0 * COUNT(o.user_principal_name) / COUNT(*), 1) AS pct
        FROM    consumers c
        LEFT JOIN {TBL} o ON o.user_principal_name = c.upn
    """).show(truncate=False)
    print("A low percentage means the two files identify people differently.")
    print("On a de-identified Viva export this will be 0 and that is expected -")
    print("hashed person IDs cannot match real UPNs. Use an identified export if")
    print("you need department breakdowns.")
else:
    print("viva_credits_weekly not loaded yet - run the Viva ingester to check the join.")